In [28]:
from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_colwidth', 100)

In [29]:
ROOT = Path.cwd().parent
RAW = ROOT / "data" / "raw"
REPORTS = ROOT / "reports"
TRAIN_NAME = "train.parquet"
BANCHMARK_Q = "benchmark_queries.parquet"
BANCHMARK_I = "benchmark_items.parquet"

for name in [TRAIN_NAME, BANCHMARK_Q, BANCHMARK_I]:
    print(name, (RAW / name).exists())

train.parquet True
benchmark_queries.parquet True
benchmark_items.parquet True


In [30]:
# Read scheme (columns' names and types)
# Question: do we have item_id in train? -> yes
for name in [TRAIN_NAME, BANCHMARK_Q, BANCHMARK_I]:
    schema = pq.read_schema(RAW / name)
    print(f'-{name}-')
    for field in schema:
        print(f"{field.name:30s} {field.type}")

-train.parquet-
search_query                   string
search_location_id             int64
search_is_delivery_search      int32
search_infm_params_text        string
search_category                int64
item_title_raw                 string
item_rating_reviews_count      double
item_rating                    double
item_price                     decimal128(27, 15)
item_microcat_id               int64
item_longitude                 decimal128(18, 15)
item_location_id               int64
item_latitude                  decimal128(17, 15)
item_is_phone_hidden           bool
item_is_message_forbidden      bool
item_infm_params_text          string
item_id                        string
item_description_raw           string
item_category_id               int64
-benchmark_queries.parquet-
query_id                       large_string
search_query                   large_string
search_location_id             int64
search_is_delivery_search      int32
search_infm_params_text        large_string
se

In [31]:
# load row data
train_cols = [f.name for f in pq.read_schema(RAW / TRAIN_NAME)
              if f.name != "item_description_raw"]
train_data = pd.read_parquet(RAW / TRAIN_NAME, columns=train_cols)
bq_data = pd.read_parquet(RAW / BANCHMARK_Q)
bi_data = pd.read_parquet(RAW / BANCHMARK_I)

print(TRAIN_NAME, train_data.shape)
print(BANCHMARK_Q, bq_data.shape)
print(BANCHMARK_I, bi_data.shape)

train.parquet (497673, 18)
benchmark_queries.parquet (2452, 6)
benchmark_items.parquet (189212, 14)


In [32]:
# check data quickly
display(train_data.head())
display(bq_data.head())
display(bi_data.head())

,search_query,search_location_id,search_is_delivery_search,search_infm_params_text,search_category,item_title_raw,item_rating_reviews_count,item_rating,item_price,item_microcat_id,item_longitude,item_location_id,item_latitude,item_is_phone_hidden,item_is_message_forbidden,item_infm_params_text,item_id,item_category_id
0,скупка телевизоров,652430,0,,114,Скупка б/у техники,91.0,4.989011,1.000000000000000,2303374,39.712818145751953,652430,54.629230499267578,True,False,"Вид услуги Место оказания услуг Первомайский пр-т, 66 Тип стоимости за услугу Работаете с юрлица...",e8b685dffe1a408e,114
1,автоподбор,640860,0,Рейтинг пользователя 4 звезды и выше,114,Автоподбор Разовый осмотр автомобиля,462.0,4.982684,3500.000000000000000,2303374,44.051009999999998,640860,56.273389999999999,False,False,"Вид услуги Место оказания услуг Нижний Новгород, Советский район, жилой комплекс Новая Кузнечиха...",92e1b0446f827b59,114
2,баня на дровах,653240,0,"Онлайн-запись Тип услуги СПА-услуги, массаж Вид услуги Красота, здоровье",114,"Баня на дровах ""Прованс"" на Цветочной",NaN,NaN,1600.000000000000000,86469,30.128784180000000,653240,59.783687590000000,False,False,"Вид услуги Красота, здоровье Место оказания услуг Санкт-Петербург, садоводческое некоммерческое ...",624846856ce81d69,114
3,изготовление госномера на авто,634670,0,"Вид услуги Оборудование, производство",114,"Изготовление дубликатов авто номеров, гос номеров",14.0,4.714286,1700.000000000000000,2303428,40.537841000000000,633570,45.424875000000000,False,False,"Вид услуги Оборудование, производство Тип услуги Производство, обработка Место оказания услуг Кр...",45b8628b9c6e7b85,114
4,укладка плитки,658430,0,Тип услуги Ремонт квартир и домов под ключ Вид услуги Ремонт и отделка,114,Ремонт и отделка квартир под ключ,1.0,5.000000,1000.000000000000000,44725,69.496444699999998,658430,56.105983729999998,False,False,Вид услуги Ремонт и отделка Тип услуги Ремонт квартир и домов под ключ Место оказания услуг ул. ...,c95a4a7daf2a967f,114


,query_id,search_query,search_location_id,search_is_delivery_search,search_infm_params_text,search_category
0,70DfDUpwjxB4lzFd,перевозки владикавказ тбилиси,649820,0,,114
1,JTrdTaZJvSiLPkXj,обзвон по базе,107620,0,Вид услуги Деловые услуги,114
2,LZCZNoVG4AFUkVRJ,липоредукция подбородка,637640,0,"Вид услуги Красота, здоровье",114
3,660ac9QVtXkRxZC3,подъемник 4 х стоечный,662810,0,,114
4,YgHcM9MVbxKnxD1e,монтаж видеодомофонов,642790,0,,114


,item_title_raw,item_rating_reviews_count,item_rating,item_price,item_microcat_id,item_longitude,item_location_id,item_latitude,item_is_phone_hidden,item_is_message_forbidden,item_infm_params_text,item_id,item_description_raw,item_category_id
0,Ремонт/выкуп компьют. и ноутбуков с выездом на дом,57.0,5.0,500.000000000000000,2097573,42.047193961004901,631060,44.228180450924000,False,False,Вид услуги Компьютерная помощь Место оказания услуг пр-т Ленина Тип стоимости за услугу Начальна...,111eb8b979577d79,"Выезд на дом в любое время, вплоть до 23:00\n\n● Диагностика устройств;\n\n● Чистка от пыли и за...",114
1,Обучение ребенка чтению,7.0,5.0,900.000000000000000,86453,49.627391820000000,631870,58.627609249999999,False,False,"Вид услуги Обучение, курсы Место оказания услуг ул. Чернышевского, 35 Тип услуги Детское развити...",75fc8e10f5a66fc4,"Дорогие родители и маленькие книголюбы! \n\nЯ, детский психолог/нейропсихолог, рада пригласить в...",114
2,Афрокудри 5+,14.0,5.0,1000.000000000000000,86467,40.936892000000000,628500,56.990307000000001,True,False,"Вид услуги Красота, здоровье Место оказания услуг городской округ Иваново, Фрунзенский район Тип...",3f6ae81704565b9c,"ВНИМАНИЕ Укладка !!!! 🥳\n\nАФРОкудри отличный вариант замены хим завивки, если хочется поменять ...",114
3,Монтаж малых архитектурных форм,2.0,5.0,5000.000000000000000,2058739,37.576183000000000,637640,55.662734999999998,False,False,"Вид услуги Строительство Место оказания услуг Севастопольский пр-т, 28к4 Тип стоимости за услугу...",0618dec37a59a21a,"Оказываем все виды услуг по сборке и монтажу игровых площадок , а так же все виды работ по устро...",114
4,Репетитор по истории 10 класс ЕГЭ,19.0,5.0,1500.000000000000000,86456,44.750473999999997,624850,48.786008000000002,False,False,"Вид услуги Обучение, курсы Место оказания услуг Волгоградская обл., Волжский, пл. имени В. И. Ле...",13da2c81574677ed,🚀 Сдай ЕГЭ по истории на 80+ за 6 месяцев!\n\nМечтаешь поступить в вуз своей мечты? ✨\n\nИстория...,114


In [33]:
# check NONEs
display(train_data.isna().mean().sort_values(ascending=False).to_frame("Train NONEs"))
display(bi_data.isna().mean().sort_values(ascending=False).to_frame("BI NONEs"))

,Train NONEs
item_rating,0.056728
item_rating_reviews_count,0.038602
item_latitude,0.000008
item_longitude,0.000008
search_location_id,0.000000
search_query,0.000000
item_title_raw,0.000000
search_category,0.000000
search_is_delivery_search,0.000000
search_infm_params_text,0.000000


,BI NONEs
item_rating,0.093181
item_rating_reviews_count,0.061386
item_description_raw,0.000185
item_longitude,0.000005
item_latitude,0.000005
item_title_raw,0.000000
item_microcat_id,0.000000
item_price,0.000000
item_is_phone_hidden,0.000000
item_location_id,0.000000


In [34]:
def as_key(s: pd.Series) -> pd.Series:
    """Превращает колонку-идентификатор в строки, пригодные для сравнения.

    Проблема: если в колонке с целыми числами есть пропуски, pandas хранит её
    как дробную, и 123 превращается в '123.0'. Тогда '123.0' != '123', и
    сравнение с другим файлом молча покажет ноль совпадений.
    Поэтому дробные колонки сначала переводим в целые (с поддержкой пропусков).
    """
    if pd.api.types.is_float_dtype(s):
        s = s.astype("Int64")
    return s.astype("string")

cols_id = ['item_id', 'query_id', 'search_location_id',
           'item_location_id', 'item_category_id', 'item_microcat_id']
for df in (train_data, bq_data, bi_data):
    for col in cols_id:
        if col in df.columns:
            df[col] = as_key(df[col])
print('корректный формат item_id:', bi_data['item_id'].str.fullmatch(r'[0-9a-f]{16}').mean())

корректный формат item_id: 1.0


In [35]:
# Пересекаются ли у нас объявления train и benchmarks по item_id?
if "item_id" in train_data.columns:
    benchmark_ids = set(bi_data["item_id"])
    train_ids = set(train_data["item_id"])
    both = benchmark_ids & train_ids
    train_data["is_item_id_in_benchmark"] = train_data["item_id"].isin(benchmark_ids)
    print(f"Уникальных объявлений train: {len(train_ids):,}")
    print(f"Уникальных объявлений benchmark: {len(benchmark_ids):,}")
    print(f"Количество пересечений: {len(both):,} {len(both) / len(train_ids):.1%}")
    print(f"Доля строк train с объявлением из benchmark: {train_data["is_item_id_in_benchmark"].mean():.1%}")
    print(f"Доля объявлений benchmark, которые были выбраны в train: {len(both) / len(benchmark_ids):.1%}")
else:
    print("У нас нет item_id")

Уникальных объявлений train: 344,825
Уникальных объявлений benchmark: 189,212
Количество пересечений: 18,142 5.3%
Доля строк train с объявлением из benchmark: 6.6%
Доля объявлений benchmark, которые были выбраны в train: 9.6%


In [36]:
# Тексты запросов benchmark уже встречаются в train?
def norm_series(s: pd.Series) -> pd.Series:
    '''Приводим текст к нормализованному виду'''
    return (s.fillna('').str.lower()
            .str.replace('ё', 'е', regex=False)
            .str.replace(r'\s+', ' ', regex=True)
            .str.strip())

train_data["normed_search_query"] = norm_series(train_data["search_query"])
bq_data["normed_search_query"] = norm_series(bq_data["search_query"])

def to_key_tuples(df, cols):
    """Объединение нескольких полей"""
    return pd.Series(list(map(tuple, df[cols].astype("string").fillna('').values)), index=df.index)

LEVELS = {
    'текст': ['normed_search_query'],
    'текст + локация': ['normed_search_query', 'search_location_id'],
    'текст + локация + доставка': [
        'normed_search_query',
        'search_location_id',
        'search_is_delivery_search'],
    'все поля поиска': [
        'normed_search_query',
        'search_location_id',
        'search_is_delivery_search',
        'search_infm_params_text',
        'search_category'],
}
result = {}
for name, cols in LEVELS.items():
    seen_keys = set(to_key_tuples(train_data, cols))
    result[name] = to_key_tuples(bq_data, cols).isin(seen_keys).mean()
display(pd.Series(result, name="доля запросов benchmark/train")
        .map('{:.1%}'.format).to_frame())

,доля запросов benchmark/train
текст,37.4%
текст + локация,6.6%
текст + локация + доставка,6.6%
все поля поиска,4.4%


In [39]:
# Можно ли обучить модель-классификатор категории?
X_text = train_data["normed_search_query"]
y = train_data["item_microcat_id"]
print('Уникальных подкатегорий: ', y.nunique())
counts = y.value_counts()
print(counts.describe()) # сколько строк на каждую подкатегорию

print("топ-10 самых частых подкатегорий")
print(counts.head(10))
print("топ-10 самых редких")
print(counts.tail(10))

cum_share = counts.cumsum() / counts.sum()
print(f"топ-10 категорий покрывают: {cum_share.iloc[9]:.1%} всего train")
print(f"топ-50 категорий покрывают: {cum_share.iloc[49]:.1%} всего train")
print(f"топ-100 категорий покрывают: {cum_share.iloc[99]:.1%} всего train")

# Сколько маленьких категорий и строк в конце
tail = counts[counts < 10]
print(f"Категорий с < 10 строк: {len(tail)} из {len(counts)}. Это {(len(tail) / len(counts)):.1%} от всех категорий")
print(f"Это всего {tail.sum()} строк train {tail.sum() / len(y):.2%} от всех строк")

Уникальных подкатегорий:  212
count          212.0
mean     2347.514151
std      3350.737376
min              1.0
25%            404.5
50%           1182.5
75%          2929.75
max          29012.0
Name: count, dtype: double[pyarrow]
топ-10 самых частых подкатегорий
item_microcat_id
1289835    29012
86470      18768
1289833    15350
86467      12680
86469      11929
2059415    10543
44730       9196
1178214     9139
2301617     8990
1289834     8753
Name: count, dtype: int64[pyarrow]
топ-10 самых редких
item_microcat_id
383        2
1287995    2
62         2
2199161    1
2171444    1
2303720    1
1144482    1
2300768    1
5529       1
2303468    1
Name: count, dtype: int64[pyarrow]
топ-10 категорий покрывают: 27.0% всего train
топ-50 категорий покрывают: 67.1% всего train
топ-100 категорий покрывают: 89.2% всего train
Категорий с < 10 строк: 13 из 212. Это 6.1% от всех категорий
Это всего 29 строк train 0.01% от всех строк


In [42]:
# Как выглядят незнакомые запросы?
unseen = bq_data.loc[~bq_data["normed_search_query"].isin(set(train_data["normed_search_query"])), "normed_search_query"]
print(f"Незнакомых запросов:{len(unseen)} из {len(bq_data["normed_search_query"])}")
print(unseen.sample(min(30, len(unseen)), random_state=67).tolist())

Незнакомых запросов:1536 из 2452
['установка диодных лент в потолок', 'сделаслелать губы', 'аниматор железный человек', 'консультация пк', 'вынос без вывоза дивана из квартиры', 'репетитор по русскому языку на жби', 'масстер класс по изготовлению цветов из зефира', 'натуральный перманент', 'услуги частника круглосуточно эвакуатора от', 'аренда кабинета краснодар любимово', 'стрижка когтей у собак с выездом на дом', 'монтаж насоса', 'услуги трактора с роторной косилкой', 'трансфер до иркутска', 'аренда барабанов', 'замена труб водоснабжения на улице', 'украшения на выписку из роддома', 'услуги по укладке ламината', 'шлифованные полы', 'изделие на 3д принтере', 'натуропат', 'продленка 40 гимн', 'шугаринг женский кайеркан', 'чистка колодцев рощино', 'брусчатка тротуарная плитка купить', 'кемерово доставка', 'бригада по ремонту домов', 'услуги психолога оторвать от алкоголя', 'настройка pin на роутер', 'массаж гимнастика малышам']
